In [ ]:
!pip install llama-index llama-index-llms-groq llama-index-embeddings-huggingface chromadb llama-index-vector-stores-chroma

In [ ]:
# Getting the source:

source_path = "/content/Corn_Leaf_Blight.md"
source_path

'/content/Corn_Leaf_Blight.md'

In [ ]:
# Configure envirnoment with Groq-API:
from google.colab import userdata

GROQ_API = userdata.get('GROQ_TOKEN')


In [ ]:
# Importing the essentials:

import os
import chromadb
from llama_index.llms.groq import Groq
from llama_index.core import Settings
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext


In [ ]:
  # Configure LLM and Embeddings Model:

Settings.llm = Groq(
      model="openai/gpt-oss-120b",
      api_key=GROQ_API
  )

# Use Hugging Face for embedding (can be replaced with OpenAI or others)
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
Settings.embed_model = embed_model

In [ ]:
# Define persistent DB location
db = chromadb.PersistentClient(path="chroma_db")

# Create or retrieve the vector collection
chroma_collection = db.get_or_create_collection("docs_collection")

In [ ]:
documents = SimpleDirectoryReader(input_files=[source_path]).load_data()

print(len(documents))
print(documents)

1
[Document(id_='828f42d9-e529-444f-96b5-51e36d67e752', embedding=None, metadata={'file_path': '/content/Corn_Leaf_Blight.md', 'file_name': 'Corn_Leaf_Blight.md', 'file_type': 'text/markdown', 'file_size': 3924, 'creation_date': '2026-10-01', 'last_modified_date': '2026-10-01'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='# Corn Leaf Blight\n\n## Overview\n\nCorn Leaf Blight is a fungal disease affecting corn plants globally. Northern Corn Leaf Blight (NCLB) is caused by Setosphaeria turcica and Southern Corn Leaf Blight (SCLB) by Bipolaris maydis. Both diseases reduce photosynthesis and grain yield when left untreated.\n\n---\n\n## Cr

In [ ]:
for doc in documents:
  print(doc)

Doc ID: 828f42d9-e529-444f-96b5-51e36d67e752
Text: # Corn Leaf Blight  ## Overview  Corn Leaf Blight is a fungal
disease affecting corn plants globally. Northern Corn Leaf Blight
(NCLB) is caused by Setosphaeria turcica and Southern Corn Leaf Blight
(SCLB) by Bipolaris maydis. Both diseases reduce photosynthesis and
grain yield when left untreated.  ---  ## Crop Information  - Crop:
Corn (Maize)...


In [ ]:
# Create parser with chunking strategy
parser = SimpleNodeParser.from_defaults(chunk_size=512, chunk_overlap=50)

# Convert documents to chunks (nodes)
nodes = parser.get_nodes_from_documents(documents)

In [ ]:
for i, node in enumerate(nodes):
  print(f"Node number: {i}")
  print(f"Node: {node}")

Node number: 0
Node: Node ID: 1ba42113-36db-46a2-a081-4e1225028d63
Text: # Corn Leaf Blight  ## Overview  Corn Leaf Blight is a fungal
disease affecting corn plants globally. Northern Corn Leaf Blight
(NCLB) is caused by Setosphaeria turcica and Southern Corn Leaf Blight
(SCLB) by Bipolaris maydis. Both diseases reduce photosynthesis and
grain yield when left untreated.  ---  ## Crop Information  - Crop:
Corn (Maize)...
Node number: 1
Node: Node ID: 6523c1ed-01b0-42e3-ac9e-1b319561f699
Text: Scout fields every 7 days for disease progression. 2. Remove
lower infected leaves if severe. 3. Improve air circulation by
controlling weeds. 4. Avoid overhead irrigation.  ---  ## Treatment
Plan  ### Day 1  Begin fungicide application if disease appears on
leaves below ear leaf or if weather favors rapid spread. Use triazole
or strobilurin fun...


In [ ]:
print(len(nodes))

2


In [ ]:
# Bind Chroma to LlamaIndex
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create storage context
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Create the vector index
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    vector_store=vector_store,
    embed_model=embed_model
)

In [ ]:
index

In [ ]:
# Create the query engine
query_engine = index.as_query_engine(similarity_top_k=1)

# Ask a question
response = query_engine.query("CAn you state the severity of Corn in day wise?")

In [ ]:
print(response)

**Day 1** – Early stage. Disease is just appearing on the lower leaves (below the ear leaf). Severity is low, but conditions that favor rapid spread (wet weather, high humidity) should trigger a fungicide application.

**Day 3** – Mid‑early stage. New lesions may be visible in the canopy. If lesions are increasing, severity moves to a moderate level and a second fungicide spray is advisable; if little change is seen, severity remains low.

**Day 5** – Mid stage. Lesion expansion is assessed to predict whether the infection will reach the ear leaf within the next week. When rapid upward movement is evident, severity is considered high and additional treatments may be needed; if growth is slow or stable, severity stays moderate to low.

**Day 7** – Late stage. The final evaluation determines the overall severity:
- If lesions have reached the flag leaf, severity is high and a final fungicide application is recommended.
- If lesions are stable or decreasing, severity is low to moderate an